In [2]:
from iFinDPy import *
import sys
import logging
sys.path.append('E:\wangzhilin\QuantumGalaxy')
from QGI.mysql import MYSQL,get_mysql
from QGI.logger import get_logger
from datetime import date,timedelta,datetime
import pandas as pd
import numpy as np


e:\ProgramData\Anaconda3\envs\QGIBG\Lib\site-packages\iFinDPy.pth


<RotatingFileHandler e:\wangzhilin\QuantumGalaxy\opt\a.log (NOTSET)>

In [1]:
from logging.handlers import RotatingFileHandler

In [3]:
logger=get_logger(fh=True,f_name='testlog2')
logger

<RootLogger root (DEBUG)>

In [4]:
mysql=get_mysql()

08-16 16:38 QGI.mysql    INFO     using username Local_Editor
08-16 16:38 QGI.mysql    INFO     accessed to database <qgdbs>


get mysql connect on database :qgdbs


In [5]:
#this is a refactoring of the process_data code using github copilot


In [7]:
region='CN'
today=date.today()-timedelta(0) if region in {'CN','HK'} else date.today()-timedelta(days=1)
test=mysql.read_query(f'select count(*) from ticker_data where date="{today}"')[0][0]
if test<=500:
    logger.error(f'No data for {today}')
    #return
logger.info(f'Processing {today}, {test} records')

08-16 16:39 QGI.mysql    INFO     got a cursor
08-16 16:39 QGI.mysql    INFO     Accepted read sql query "select count(*) from ticker_data where date="2023-08-16""
08-16 16:39 root         INFO     Processing 2023-08-16, 6974 records


In [8]:
def query_data(mysql,today):
    '''get ticker data from today-370 to today,and process to dataframe'''
    start_date=(today-timedelta(370)).__format__('%Y-%m-%d')
    end_date=today.__format__("%Y-%m-%d")
    #sql="select a.code, b.date,b.close_price from ticker_info a inner join ticker_data b on a.code = b.code where region ='%s' and b.date>'%s' and a.status=1"%(region,start_date)
    sql="select a.code, b.date,b.close_price from ticker_info a inner join ticker_data b on a.code = b.code inner join qg_indicator_info c on c.qg_code=a.code where  b.date>'%s' and b.date<='%s' and a.status=1 and c.need_process=1 "%(start_date,end_date)
    result=mysql.read_query(sql)
    sql="select c.qg_code, b.date,b.value from edb_info a inner join wind_edb_data b on a.wind_code = b.code inner join qg_indicator_info c on c.qg_code=a.code where  b.date>'%s' and b.date<='%s'  and c.need_process=1 "%(start_date,end_date)
    
    
    result+=mysql.read_query(sql)
    df=pd.DataFrame(result,columns=['code','date','close_price'])
    df['date']=pd.to_datetime(df['date'])
    groups=df.groupby('code')
    df_list=[]
    for code,t in groups:
        if t.shape[0]>1:
            df=t.set_index('date')
            df1=pd.DataFrame(data={df.iloc[1]['code']:df['close_price']},index=df.index)
            if df1.index[-1].__format__("%Y-%m-%d")==today.__format__("%Y-%m-%d"):
                df_list.append(df1)
    print(len(df_list))
    result=pd.concat(df_list,axis=1)
    result.replace({None:np.nan},inplace=True)
    result.fillna(method='pad',limit=10,inplace=True)
    result=result.astype(float)
    return result

In [9]:
result=query_data(mysql,today)
result

08-16 16:40 QGI.mysql    INFO     got a cursor
08-16 16:40 QGI.mysql    INFO     Accepted read sql query "select a.code, b.date,b.close_price from ticker_info a inner join ticker_data b on a.code = b.code inner join qg_indicator_info c on c.qg_code=a.code where  b.date>'2022-08-11' and b.date<='2023-08-16' and a.status=1 and c.need_process=1 "
08-16 16:41 QGI.mysql    INFO     got a cursor
08-16 16:41 QGI.mysql    INFO     Accepted read sql query "select c.qg_code, b.date,b.value from edb_info a inner join wind_edb_data b on a.wind_code = b.code inner join qg_indicator_info c on c.qg_code=a.code where  b.date>'2022-08-11' and b.date<='2023-08-16'  and c.need_process=1 "


6987


,000001.SZ,000002.SZ,000004.SZ,000005.SZ,000006.SZ,000007.SZ,000008.SZ,000009.SZ,000010.SZ,000011.SZ,...,WEBF.HK,WH.CZC,WHAF.HK,WHLF.HK,WXBF.HK,Y.DCE,YZCF.HK,ZAOF.HK,ZC.CZC,ZJMF.HK
date,,,,,,,,,,,,,,,,,,,,,
2022-08-12,12.38,16.38,9.68,1.79,4.13,8.01,2.44,16.27,3.48,10.32,...,0.0,0.0,NaN,0.00,0.00,10154.0,28.52,20.94,850.2,9.43
2022-08-15,12.11,16.29,9.88,1.81,4.09,8.16,2.45,16.57,3.46,10.19,...,0.0,3131.0,NaN,0.00,70.20,9968.0,28.97,20.95,860.0,0.00
2022-08-16,12.13,16.53,10.19,1.81,4.16,8.12,2.45,16.10,3.51,10.32,...,0.0,3145.0,NaN,0.00,70.49,9974.0,0.00,0.00,860.0,9.23
2022-08-17,12.40,16.88,9.96,1.84,4.17,7.99,2.50,16.55,3.53,10.41,...,0.0,0.0,NaN,28.62,0.00,10048.0,27.73,0.00,858.8,0.00
2022-08-18,12.25,16.79,9.69,1.82,4.10,7.84,2.47,17.16,3.44,10.23,...,0.0,0.0,NaN,0.00,72.10,9466.0,27.92,0.00,850.0,9.05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-08-10,12.16,14.75,18.69,1.44,5.30,5.09,2.43,11.19,2.68,10.31,...,0.0,0.0,0.0,0.00,43.92,8168.0,0.00,0.00,0.0,0.00
2023-08-11,11.89,14.60,17.70,1.41,5.83,5.19,2.39,10.96,2.60,10.83,...,0.0,0.0,0.0,0.00,0.00,8168.0,12.32,0.00,0.0,0.00
2023-08-14,11.67,14.40,18.59,1.43,5.25,5.18,2.40,11.01,2.59,10.28,...,0.0,0.0,0.0,0.00,0.00,8146.0,0.00,0.00,0.0,0.00


In [10]:
result.truncate(after=today-timedelta(365))

,000001.SZ,000002.SZ,000004.SZ,000005.SZ,000006.SZ,000007.SZ,000008.SZ,000009.SZ,000010.SZ,000011.SZ,...,WEBF.HK,WH.CZC,WHAF.HK,WHLF.HK,WXBF.HK,Y.DCE,YZCF.HK,ZAOF.HK,ZC.CZC,ZJMF.HK
date,,,,,,,,,,,,,,,,,,,,,
2022-08-12,12.38,16.38,9.68,1.79,4.13,8.01,2.44,16.27,3.48,10.32,...,0.0,0.0,NaN,0.0,0.00,10154.0,28.52,20.94,850.2,9.43
2022-08-15,12.11,16.29,9.88,1.81,4.09,8.16,2.45,16.57,3.46,10.19,...,0.0,3131.0,NaN,0.0,70.20,9968.0,28.97,20.95,860.0,0.00
2022-08-16,12.13,16.53,10.19,1.81,4.16,8.12,2.45,16.10,3.51,10.32,...,0.0,3145.0,NaN,0.0,70.49,9974.0,0.00,0.00,860.0,9.23


In [31]:
def compute(df,today):
    processed_df = pd.DataFrame(
        data={
            'date': today,
            'chg1d': compute_chg_from_days(df, today, 1),
            'chg3d': compute_chg_from_days(df, today, 3),
            'chg2w': compute_chg_from_days(df, today, 14),
            'chg1m': compute_chg_from_days(df, today, 30),
            'chg3m': compute_chg_from_days(df, today, 91),
            'chg6m': compute_chg_from_days(df, today, 182),
            'chg1y': compute_chg_from_days(df, today, 365),
            'stdchg7d': compute_std_chg_from_days(df, today, 7),
            'stdchg1m': compute_std_chg_from_days(df, today, 30),
            'stdchg3m': compute_std_chg_from_days(df, today, 90),
            'stdchg': compute_stdchg60(df, today),
            'change_mean':compute_meanchg60(df,today)
        },
        index=df.columns,
    )
    return processed_df

,000001.SZ,000002.SZ,000004.SZ,000005.SZ,000006.SZ,000007.SZ,000008.SZ,000009.SZ,000010.SZ,000011.SZ,...,WEBF.HK,WH.CZC,WHAF.HK,WHLF.HK,WXBF.HK,Y.DCE,YZCF.HK,ZAOF.HK,ZC.CZC,ZJMF.HK
date,,,,,,,,,,,,,,,,,,,,,
2022-08-12,12.38,16.38,9.68,1.79,4.13,8.01,2.44,16.27,3.48,10.32,...,0.0,0.0,NaN,0.00,0.00,10154.0,28.52,20.94,850.2,9.43
2022-08-15,12.11,16.29,9.88,1.81,4.09,8.16,2.45,16.57,3.46,10.19,...,0.0,3131.0,NaN,0.00,70.20,9968.0,28.97,20.95,860.0,0.00
2022-08-16,12.13,16.53,10.19,1.81,4.16,8.12,2.45,16.10,3.51,10.32,...,0.0,3145.0,NaN,0.00,70.49,9974.0,0.00,0.00,860.0,9.23
2022-08-17,12.40,16.88,9.96,1.84,4.17,7.99,2.50,16.55,3.53,10.41,...,0.0,0.0,NaN,28.62,0.00,10048.0,27.73,0.00,858.8,0.00
2022-08-18,12.25,16.79,9.69,1.82,4.10,7.84,2.47,17.16,3.44,10.23,...,0.0,0.0,NaN,0.00,72.10,9466.0,27.92,0.00,850.0,9.05
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2023-08-10,12.16,14.75,18.69,1.44,5.30,5.09,2.43,11.19,2.68,10.31,...,0.0,0.0,0.0,0.00,43.92,8168.0,0.00,0.00,0.0,0.00
2023-08-11,11.89,14.60,17.70,1.41,5.83,5.19,2.39,10.96,2.60,10.83,...,0.0,0.0,0.0,0.00,0.00,8168.0,12.32,0.00,0.0,0.00
2023-08-14,11.67,14.40,18.59,1.43,5.25,5.18,2.40,11.01,2.59,10.28,...,0.0,0.0,0.0,0.00,0.00,8146.0,0.00,0.00,0.0,0.00


In [26]:
# give a decorateor to handle the exception
def handle_processing_exception(fn):
    def wrapper(*args, **kwargs):
        try:
            return fn(*args, **kwargs)
        except Exception as e:
            #logger this error with the function name and stack trace
            logger.error(f'Error in {fn.__name__}: {e}', exc_info=True)
            return None

    return wrapper

In [38]:
@handle_processing_exception
def compute_stdchg60(df,today):

        df=np.log(df)
        df=df.truncate(after=today)
        stdchg60=(df-df.shift(1)).iloc[-60:].std()
        return stdchg60
@handle_processing_exception
def compute_meanchg60(df,today):
        
                df=np.log(df)
                df=df.truncate(after=today)
                meanchg60=(df-df.shift(1)).iloc[-60:].mean()
                return meanchg60
@handle_processing_exception            
def compute_std_chg_from_days(df,today,days):
    #days=7,30,90

        df=np.log(df)
        df=df.truncate(after=today)
        after=df.iloc[-1]
        former=df.truncate(after=today-timedelta(days)).iloc[-1]
        stdchg60=(df-df.shift(1)).iloc[-60:,:].std()
        
        result=(after-former)/(np.sqrt(days)*stdchg60)
        return result
@handle_processing_exception
def compute_chg_from_days(df,today,days):
    #df=df.truncate(after=today).iloc[-1]
    former=df.truncate(after=today-timedelta(days)).iloc[-1]
    df=df.truncate(after=today).iloc[-1]
    return (df-former)/former

In [39]:
compute_chg_from_days(result,today,1)

000001.SZ    0.000000
000002.SZ    0.017495
000004.SZ   -0.033683
000005.SZ    0.013514
000006.SZ    0.087037
               ...   
Y.DCE        0.017032
YZCF.HK           NaN
ZAOF.HK           NaN
ZC.CZC            NaN
ZJMF.HK           NaN
Length: 6987, dtype: float64

In [40]:
compute(result,today)

e:\ProgramData\Anaconda3\envs\QGIBG\lib\site-packages\pandas\core\internals\blocks.py:402: RuntimeWarning: invalid value encountered in log
  result = func(self.values, **kwargs)
e:\ProgramData\Anaconda3\envs\QGIBG\lib\site-packages\pandas\core\internals\blocks.py:402: RuntimeWarning: divide by zero encountered in log
  result = func(self.values, **kwargs)


,date,chg1d,chg3d,chg2w,chg1m,chg3m,chg6m,chg1y,stdchg7d,stdchg1m,stdchg3m,stdchg,change_mean
000001.SZ,2023-08-16,0.000000,-0.013457,-0.024938,0.037135,-0.060849,-0.141917,-0.032976,-1.026869,0.491150,-0.488226,0.013554,-0.000393
000002.SZ,2023-08-16,0.017495,-0.004110,-0.034529,0.060540,-0.016238,-0.161476,-0.120387,-0.238349,0.618358,-0.033327,0.017355,0.000500
000004.SZ,2023-08-16,-0.033683,-0.011299,-0.110772,-0.167063,0.702335,0.734390,0.717370,-1.077450,-0.569807,0.878652,0.058571,0.006940
000005.SZ,2023-08-16,0.013514,0.063830,0.034483,0.250000,0.020408,-0.175824,-0.171271,0.598367,1.349576,0.023357,0.030187,0.002514
000006.SZ,2023-08-16,0.087037,0.006861,0.135397,0.346330,0.243644,0.053860,0.411058,2.255992,1.947140,0.824269,0.027884,0.003991
...,...,...,...,...,...,...,...,...,...,...,...,...,...
Y.DCE,2023-08-16,0.017032,0.023506,0.028796,0.043435,0.169558,-0.030612,-0.161821,0.586909,0.524327,1.121126,0.014805,0.002988
YZCF.HK,2023-08-16,NaN,-1.000000,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ZAOF.HK,2023-08-16,NaN,NaN,-1.000000,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
ZC.CZC,2023-08-16,NaN,NaN,NaN,NaN,NaN,NaN,-1.000000,NaN,NaN,NaN,NaN,NaN


In [64]:
compute_chg_rate_from_days(result,today,2)

000001.SZ    0.005141
000002.SZ    0.009722
000004.SZ   -0.058634
000005.SZ    0.048951
000006.SZ    0.118095
               ...   
UR.CZC       0.020873
V.DCE       -0.011216
WH.CZC            NaN
Y.DCE        0.026271
ZC.CZC            NaN
Length: 5206, dtype: float64